# Lab 3 — Chunking strategies (EP10)

The book: chunking is **one of the highest-impact decisions** in a RAG system (§16.4).
In this lab we chunk the same library four ways and measure each with Lab 2's hit@3, on **20 questions**

**What you'll do**
1. Check the chunk-count formula (Equation 16.19) against Lab 1's code
2. Fixed size, 512 and 128 tokens
3. **Structure-aware** — split at the book's headings (§16.4.3)
4. **Parent-child** — search small chunks, send big ones (§16.4.4)
5. Go back to the question we've missed since Lab 1

This lab only uses the embedding model, and searches with dense only, so the chunking is the only thing that changes

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.4 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep10-lab3-chunking.en.ipynb`** (the same code as in the video).

## 0. Setup

The same as Lab 1: Python 3.10+ or Colab, and `OPENROUTER_API_KEY` (Colab: 🔑 Secrets · your own machine: a `.env` file).
This lab embeds four libraries — it costs under 1 US cent.
How to set up the key, and why we chose these models, is in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"

## 1. The book, 20 questions, and the scorer

The first cell is Lab 1's code: the client, `embed`, loading pages 292–374, and `n_tokens` to count tokens.
**One difference from Lab 1:** every chunk keeps `pages` as a list — a chunk split at headings can span two pages

In [ ]:
import os, re, math, urllib.request
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
tok = Tokenizer.from_pretrained("google/gemma-4-26b-a4b-it")

def n_tokens(text):
    return len(tok.encode(text, add_special_tokens=False).ids)

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64], extra_body=EMBED_EXTRA)
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"source": "hitchhikers-guide-v1.pdf", "page": n,
          "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]

In [ ]:
TESTS = [
    ("What is PLAID?", {301}),
    ("What is HyDE?", {303}),
    ("What does FAISS stand for?", {298}),
    ("How can I search millions of vectors quickly?", {298}),
    ("Which method compares every query token with every document token?", {300}),
    ("How do I make keyword search understand synonyms?", {299}),
    ("When should I fine-tune a model instead of looking things up?", {296}),
    ("Which chunks should I give the model if small ones retrieve better but lack context?", {302, 303}),
    ("How do I combine keyword search and embedding search?", {298}),
    ("What analogy does the textbook use to explain RAG?", {295}),
    ("How does the textbook add randomness to retry waits?", {357}),
    ("What does FIFO mean for conversation history?", {345}),
    ("Where should the unchanging part of a prompt go so caching works?", {358}),
    ("When should a person approve an agent's action?", {355}),
    ("How does Graph RAG find groups of related entities?", {305}),
    ("Which chunking method splits where the topic changes?", {302}),
    ("Which frameworks are compared in Table 18.1?", {359}),
    ("What is the difference between episodic and semantic memory?", {321, 322}),
    ("What does an evaluator-optimizer workflow do?", {370, 371}),
    ("How does the harness notice an agent repeating the same tool call?", {357, 363}),
]
QV = embed([q for q, _ in TESTS])

In [ ]:
def evaluate(name, search_chunks, parent_of=None, parents=None):
    E = embed([c["text"] for c in search_chunks])
    returned = parents if parents else search_chunks
    hits, context = 0, []
    for (q, gold), qv in zip(TESTS, QV):
        order = list(np.argsort(-(E @ qv)))
        # ✏️ TODO 1 (▶ video 1:46): turn the ranking of children into a ranking of parents, removing repeated parents (keep the order)
        if parent_of:
            order = ___
        top3 = [returned[i] for i in order[:3]]
        # ✏️ TODO 2 (▶ video 1:59): it's a hit if any top-3 chunk has a page in gold (intersect sets)
        hits += ___
        context.append(sum(n_tokens(c["text"]) for c in top3))
    print(f"{name:28s} {len(search_chunks):4d} chunks · hit@3 {hits:2d}/{len(TESTS)} · "
          f"context ≈ {np.mean(context):,.0f} tokens")
    return E

## 2. Fixed size + the chunk-count formula

$$N_{\text{chunks}}=\left\lceil\frac{L-O}{W-O}\right\rceil\qquad(16.19)$$

Lab 1's `chunk_page` — now returning `pages` as a list

In [ ]:
def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"source": page["source"], "pages": [page["page"]],
             "text": tok.decode(ids[start:start + W])}
            for start in range(0, max(len(ids) - O, 1), W - O)]

# ✏️ TODO 3 (▶ video 2:26): the number of chunks per Equation 16.19: ceil((L − O) / (W − O))

def expected_chunks(L, W, O):
    return ___

for p in pages[:5]:
    L = n_tokens(p["text"])
    print(p["page"], "L =", L, "· formula:", expected_chunks(L, 512, 64), "· ours:", len(chunk_page(p)))

In [ ]:
fixed512 = [c for p in pages for c in chunk_page(p)]
E_fixed512 = evaluate("fixed 512 / overlap 64", fixed512)

In [ ]:
fixed128 = [c for p in pages for c in chunk_page(p, W=128, O=16)]
E_fixed128 = evaluate("fixed 128 / overlap 16", fixed128)
print(repr(fixed128[12]["text"]))

## 3. Structure-aware chunking (§16.4.3)

The book is divided by headings: `Chapter 16`, `16.4.1 Fixed-Size …` — we start a new chunk **at every heading**,
and when a chunk would go over 512 tokens · we cut between lines, never mid-line · a chunk can span two pages

In [ ]:
HEADING = re.compile(r"^(Chapter \d+|\d+\.\d+(\.\d+)?\s+[A-Z].{2,90})$")

def lines_of(pages):
    for p in pages:
        lines = p["text"].split("\n")
        if lines[-1].strip().isdigit():          # the page number at the bottom
            lines = lines[:-1]
        for line in lines:
            yield line, p["page"]

# ✏️ TODO 4 (▶ video 4:18): start a new chunk when there's none yet, the line is a heading, or adding it would exceed max_tokens

def structure_chunks(pages, max_tokens=512):
    chunks, cur = [], None
    for line, page in lines_of(pages):
        n = n_tokens(line)
        if ___:
            if cur:
                chunks.append(cur)
            cur = {"source": "hitchhikers-guide-v1.pdf", "pages": [], "lines": [], "tokens": 0}
        cur["lines"].append(line)
        cur["pages"].append(page)
        cur["tokens"] += n
    chunks.append(cur)
    for c in chunks:
        c["text"] = "\n".join(c.pop("lines"))
        c["pages"] = sorted(set(c["pages"]))
    return chunks

In [ ]:
sections = structure_chunks(pages)
print(len(sections), "chunks")
for c in sections[3:7]:
    print(c["pages"], c["tokens"], "tokens ·", c["text"].split("\n")[0])
E_sections = evaluate("structure (≤ 512)", sections)

## 4. Parent-child (§16.4.4)

**Children** = 128-token chunks (overlap 16) cut from each **parent** = a section chunk from part 3.
Search the children (precise) → send the parents to the model (full context)

In [ ]:
# ✏️ TODO 5 (▶ video 5:29): give every child its parent's pages and record the parent's index in parent_of
children, parent_of = [], []
for pi, parent in enumerate(sections):
    for child in chunk_page({"source": parent["source"], "page": 0, "text": parent["text"]}, W=128, O=16):
        child["pages"] = ___
        children.append(child)
        parent_of.append(___)
E_child = evaluate("parent-child (128 → section)", children, parent_of, sections)

## 5. The question we've missed since Lab 1

With our best method (parent-child) — at which rank does the section containing "library card" come up?

In [ ]:
question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
order = list(dict.fromkeys(parent_of[i] for i in np.argsort(-(E_child @ embed([question])[0]))))
print("library card section at rank", [r for r, pi in enumerate(order, 1) if "library card" in sections[pi]["text"]])

## Summary

| Chunking | Chunks | hit@3 (20 questions) | Context per question |
|---|---|---|---|
| Fixed 512 / overlap 64 (Lab 1) | 141 | 12 | ≈ 1,050 tokens |
| Fixed 128 / overlap 16 | 431 | 14 | ≈ 340 tokens — but chunks start mid-sentence |
| Split at headings (≤ 512) | 184 | 16 | ≈ 440 tokens |
| Parent-child (128 children → section parents) | 461 children | **17** | ≈ 740 tokens |

- Equation 16.19 matches our chunk counts on every page
- Lab 1's long question: the "library card" section is at rank 17 — no chunking helps → transform the query before searching (EP11)

**Exercises**
1. Try `chunk_page(p, W=512, O=0)` (no overlap) — on our 20 questions, no overlap scores **higher** than overlap! Find which questions differ and think about why (overlap helps when a key sentence is cut at a boundary — do our questions have that case?)
2. Use Lab 2's hybrid instead of dense with the section chunks — what's hit@3?
3. Try `structure_chunks(pages, max_tokens=256)`

**Next — Lab 4:** build the same RAG with LangChain and compare it with our own code, piece by piece